# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [ ]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

Dim_Klant (SCD TYPE 2) ROBBERT

Dim_Accessoire (SCD TYPE 1) ROBBERT

Dim_Datum ROBBERT

Fct_Verkoop ROBBERT


Dim_Fiets (SCD Type 2) MEES

In [ ]:
logger.info("Ophalen Fiets uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, naam, adres, plaats
FROM (
    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Inkoop_Fiets f
    JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Verkoop_Fiets f
    JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Onderhoud_Fiets f
    JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
)
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
    fiets_sk
FROM Dim_Fiets
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

for row in source_data:
    (fietsnr, soort, merk, type_, kleur,
     fabrikantnr, fab_naam, fab_adres, fab_plaats) = row

    now = datetime.now()

    if fietsnr not in dwh_data:
        # ➜ NIEUW
        logger.info(f"Nieuwe fiets: {fietsnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (fietsnr, soort, merk, type_, kleur,
              fabrikantnr, fab_naam, fab_adres, fab_plaats,
              now))

    else:
        # ➜ BESTAAND → check wijzigingen
        dwh_row = dwh_data[fietsnr]

        (_, d_soort, d_merk, d_type, d_kleur,
         d_fabnr, d_fabnaam, d_fabadres, d_fabplaats,
         fiets_sk) = dwh_row

        if (soort, merk, type_, kleur,
            fabrikantnr, fab_naam, fab_adres, fab_plaats) != \
           (d_soort, d_merk, d_type, d_kleur,
            d_fabnr, d_fabnaam, d_fabadres, d_fabplaats):

            logger.info(f"Wijziging fiets: {fietsnr}")

            # 1. Sluit oude record
            dwh_cursor.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, fiets_sk))

            # 2. Insert nieuwe versie
            dwh_cursor.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (fietsnr, soort, merk, type_, kleur,
                  fabrikantnr, fab_naam, fab_adres, fab_plaats,
                  now))

dwh_conn.commit()
logger.info("Fiets dimension geüpdatet in DWH.")

Dim_Monteur (SCD Type 1) MEES

Fct_Onderhoud MEES

Fct_Inkoop MEES